# Fanout and Prompt Testing

The purpose of this notebook is to find the current SOTA model that performs the best on Sentiment140.

# mistralai/mistral-7b-instruct-v0.1

In [1]:
import os
import json
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from google.colab import userdata

OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
HTTP_REFERER = "https://github.com/bencejdanko/slm-peft-tuning-sentiment140"

MODELS_TO_TEST = [
    "mistralai/mistral-7b-instruct-v0.1",
]

def call_openrouter(model, prompt, temperature=0.1):
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "HTTP-Referer": HTTP_REFERER,
        "Content-Type": "application/json"
    }
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "max_tokens": 10
    }
    try:
        response = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=15)
        response.raise_for_status()
        return response.json()['choices'][0]['message']['content'].strip().lower()
    except Exception as e:
        return f"error: {str(e)}"

In [2]:
from datasets import load_dataset, DatasetDict, Dataset
import pandas as pd

SEED=15179996

# 5k train, 1k for testing
def prepare_dataset(dataset_name="bdanko/sentiment140", train_size=5000, test_size=1000):
    print(f"Loading dataset {dataset_name}...")
    dataset = load_dataset(dataset_name, split="train")
    df = dataset.to_pandas()

    # negative sentiment swapped to 1
    df['sentiment'] = df['sentiment'].replace(4, 1)

    train_df = df.sample(n=train_size, weights=df['sentiment'].map({0: 0.5, 1: 0.5}), random_state=SEED)
    remaining_df = df.drop(train_df.index)

    test_df_neg = remaining_df[remaining_df['sentiment'] == 0].sample(n=test_size // 2, random_state=SEED)
    test_df_pos = remaining_df[remaining_df['sentiment'] == 1].sample(n=test_size // 2, random_state=SEED)
    test_df = pd.concat([test_df_neg, test_df_pos]).sample(frac=1, random_state=SEED)

    return DatasetDict({
        "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
        "test": Dataset.from_pandas(test_df.reset_index(drop=True))
    })

dataset = prepare_dataset()
print(dataset)


Loading dataset bdanko/sentiment140...


README.md:   0%|          | 0.00/420 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/113M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1600000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'date', 'user', 'sentiment', 'query'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['text', 'date', 'user', 'sentiment', 'query'],
        num_rows: 1000
    })
})


In [ ]:
NUM_EVAL_SAMPLES = 500
CONCURRENT_THREADS = 10
SEED=15179996 # reproducible split to get our testing set

eval_subset = dataset["test"].shuffle(seed=SEED).select(range(NUM_EVAL_SAMPLES))

def evaluate_model_concurrently(model_name):
    print(f"Starting evaluation for: {model_name}")
    results = []

    with ThreadPoolExecutor(max_workers=CONCURRENT_THREADS) as executor:
        future_to_sample = {
            executor.submit(
                call_openrouter,
                model_name,
                f"Classify the sentiment of this text as 'Positive' or 'Negative'. Text: {sample['text']}\nSentiment:"
            ): sample for sample in eval_subset
        }

        for future in tqdm(as_completed(future_to_sample), total=NUM_EVAL_SAMPLES, desc=model_name):
            sample = future_to_sample[future]
            prediction = future.result()

            # Map prediction to label (0: Negative, 1: Positive)
            pred_label = 1 if "positive" in prediction else 0 if "negative" in prediction else -1

            results.append({
                "true_label": sample['sentiment'],
                "pred_label": pred_label,
                "raw_output": prediction
            })

    return results

fanout_results = {}
for model in MODELS_TO_TEST:
    fanout_results[model] = evaluate_model_concurrently(model)

Starting evaluation for: mistralai/mistral-7b-instruct-v0.1


mistralai/mistral-7b-instruct-v0.1:   0%|          | 0/500 [00:00<?, ?it/s]

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

comparison_data = []

for model, results in fanout_results.items():
    y_true = [r['true_label'] for r in results if r['pred_label'] != -1]
    y_pred = [r['pred_label'] for r in results if r['pred_label'] != -1]

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    coverage = len(y_true) / NUM_EVAL_SAMPLES # Tracks how many times model gave a valid 'pos/neg' answer

    comparison_data.append({
        "Model": model,
        "Accuracy": round(acc, 4),
        "F1": round(f1, 4),
        "Valid Responses": f"{len(y_true)}/{NUM_EVAL_SAMPLES}"
    })

comparison_df = pd.DataFrame(comparison_data).sort_values(by="F1", ascending=False)
print("\n--- Fanout Testing Results ---")
print(comparison_df.to_string(index=False))


--- Fanout Testing Results ---
                             Model  Accuracy     F1 Valid Responses
mistralai/mistral-7b-instruct-v0.1    0.8402 0.7669         194/500


In [ ]:
fanout_results['google/gemma-4-31b-it'] = evaluate_model_concurrently('google/gemma-4-31b-it')

Starting evaluation for: google/gemma-4-31b-it


google/gemma-4-31b-it:   0%|          | 0/500 [00:00<?, ?it/s]

In [ ]:
from sklearn.metrics import accuracy_score

# Sample data provided for google/gemma-4-31b-it
gemma_results = fanout_results.get('google/gemma-4-31b-it', [])

# Filter for valid binary predictions (0 or 1)
# This excludes 'neutral' or errors marked as -1
valid_evals = [r for r in gemma_results if r['pred_label'] in [0, 1]]

if valid_evals:
    y_true = [r['true_label'] for r in valid_evals]
    y_pred = [r['pred_label'] for r in valid_evals]

    # Calculate accuracy
    accuracy = accuracy_score(y_true, y_pred)

    print(f"Model: google/gemma-4-31b-it")
    print(f"Total Samples: {len(gemma_results)}")
    print(f"Valid Binary Predictions: {len(valid_evals)}")
    print(f"Accuracy: {accuracy:.4f}")
else:
    print("No valid binary predictions found to calculate accuracy.")

Model: google/gemma-4-31b-it
Total Samples: 500
Valid Binary Predictions: 489
Accuracy: 0.8098


In [18]:
model = 'openai/gpt-oss-20b'

fanout_results[model] = evaluate_model_concurrently(model)

# Sample data provided for google/gemma-4-31b-it
gemma_results = fanout_results.get(model, [])

# Filter for valid binary predictions (0 or 1)
# This excludes 'neutral' or errors marked as -1
valid_evals = [r for r in gemma_results if r['pred_label'] in [0, 1]]

if valid_evals:
    y_true = [r['true_label'] for r in valid_evals]
    y_pred = [r['pred_label'] for r in valid_evals]

    # Calculate accuracy
    accuracy = accuracy_score(y_true, y_pred)

    print(f"Model: {model}")
    print(f"Total Samples: {len(gemma_results)}")
    print(f"Valid Binary Predictions: {len(valid_evals)}")
    print(f"Accuracy: {accuracy:.4f}")
else:
    print("No valid binary predictions found to calculate accuracy.")

Starting evaluation for: openai/gpt-oss-20b


openai/gpt-oss-20b:   0%|          | 0/500 [00:00<?, ?it/s]

Model: openai/gpt-oss-20b
Total Samples: 500
Valid Binary Predictions: 61
Accuracy: 0.8361


In [20]:
model = 'mistralai/mistral-small-3.2-24b-instruct'

fanout_results[model] = evaluate_model_concurrently(model)

# Sample data provided for google/gemma-4-31b-it
gemma_results = fanout_results.get(model, [])

# Filter for valid binary predictions (0 or 1)
# This excludes 'neutral' or errors marked as -1
valid_evals = [r for r in gemma_results if r['pred_label'] in [0, 1]]

if valid_evals:
    y_true = [r['true_label'] for r in valid_evals]
    y_pred = [r['pred_label'] for r in valid_evals]

    # Calculate accuracy
    accuracy = accuracy_score(y_true, y_pred)

    print(f"Model: {model}")
    print(f"Total Samples: {len(gemma_results)}")
    print(f"Valid Binary Predictions: {len(valid_evals)}")
    print(f"Accuracy: {accuracy:.4f}")
else:
    print("No valid binary predictions found to calculate accuracy.")

Starting evaluation for: mistralai/mistral-small-3.2-24b-instruct


mistralai/mistral-small-3.2-24b-instruct:   0%|          | 0/500 [00:00<?, ?it/s]

Model: mistralai/mistral-small-3.2-24b-instruct
Total Samples: 500
Valid Binary Predictions: 307
Accuracy: 0.8208


In [21]:
model = 'mistralai/ministral-14b-2512'

fanout_results[model] = evaluate_model_concurrently(model)

# Sample data provided for google/gemma-4-31b-it
gemma_results = fanout_results.get(model, [])

# Filter for valid binary predictions (0 or 1)
# This excludes 'neutral' or errors marked as -1
valid_evals = [r for r in gemma_results if r['pred_label'] in [0, 1]]

if valid_evals:
    y_true = [r['true_label'] for r in valid_evals]
    y_pred = [r['pred_label'] for r in valid_evals]

    # Calculate accuracy
    accuracy = accuracy_score(y_true, y_pred)

    print(f"Model: {model}")
    print(f"Total Samples: {len(gemma_results)}")
    print(f"Valid Binary Predictions: {len(valid_evals)}")
    print(f"Accuracy: {accuracy:.4f}")
else:
    print("No valid binary predictions found to calculate accuracy.")

Starting evaluation for: mistralai/ministral-14b-2512


mistralai/ministral-14b-2512:   0%|          | 0/500 [00:00<?, ?it/s]

Model: mistralai/ministral-14b-2512
Total Samples: 500
Valid Binary Predictions: 333
Accuracy: 0.8108


In [41]:
system_instruction = (
"""You are an expert Linguistic Forensic Analyst specializing in early-era social media discourse and sentiment polarity.
Your goal is to decode the emotional intent of high-noise, informal micro-blogging text (Twitter/X).

Linguistic Heuristics for Sentiment140:
1. Punctuation Intensity: Repeated exclamation marks or question marks often amplify the underlying emotion, whether joy or frustration.
2. Character Elongation: Elongation of words indicates high emotional arousal; context determines if this is positive (excitement) or negative (desperation).
3. Sarcasm Detection: Be wary of phrases that are contextually negative despite having a positive token.
4. Token Frequency: The presence of hyperlinks links often correlates with news/neutrality, but in binary classification, these often lean toward 'Negative' if they involve complaints or 'Positive' if sharing joy.
5. Emoticons & Slang: Standard 2009-era emoticons (e.g., ':D', ':(', '<3') are high-weight indicators.

Analyze the combination of syntax, slang, and punctuation. Output ONLY the integer 1 (Positive) or 0 (Negative).
""")

def evaluate_model_concurrently(model_name):
    print(f"Starting evaluation for: {model_name}")
    results = []

    with ThreadPoolExecutor(max_workers=CONCURRENT_THREADS) as executor:
        future_to_sample = {
            executor.submit(
                call_openrouter,
                model_name,
                f"{system_instruction}\n\n"
                f"""\
Sentiment Polarity Request: Analyze the emotional trajectory of the following micro-blog post.
--------------------------
Post: "{sample['text']}"
--------------------------
Evaluation Guidelines:
- Positive (1): Contentment, humor, excitement, gratitude, or affectionate interaction.
- Negative (0): Boredom, sadness, anger, illness, technical frustration, or loneliness.

Predict the survival of the sentiment as 1 (Positive) or 0 (Negative). Output ONLY the digit.
Prediction:"""
            ): sample for sample in eval_subset
        }

        for future in tqdm(as_completed(future_to_sample), total=NUM_EVAL_SAMPLES, desc=model_name):
            sample = future_to_sample[future]
            prediction = future.result()

            # Map prediction to label (0: Negative, 1: Positive)
            pred_label = 1 if "positive" in prediction else 0 if "negative" in prediction else -1

            results.append({
                "true_label": sample['sentiment'],
                "pred_label": pred_label,
                "raw_output": prediction
            })

    return results


In [29]:
model = 'mistralai/ministral-14b-2512'

fanout_results[model] = evaluate_model_concurrently(model)

Starting evaluation for: mistralai/ministral-14b-2512


mistralai/ministral-14b-2512:   0%|          | 0/500 [00:00<?, ?it/s]

No valid binary predictions found to calculate accuracy.


In [31]:
# Correcting pred_label based on raw_output digits '0' or '1'
processed_results = []
for r in results:
    raw = str(r['raw_output']).strip()

    # Direct digit mapping (SOTA for structured output)
    if '1' in raw and '0' not in raw:
        r['pred_label'] = 1
    elif '0' in raw and '1' not in raw:
        r['pred_label'] = 0
    else:
        # Fallback for models that might still include text
        r['pred_label'] = 1 if "positive" in raw.lower() else 0 if "negative" in raw.lower() else -1

    processed_results.append(r)

# Filter for valid binary predictions
valid_evals = [r for r in processed_results if r['pred_label'] in [0, 1]]

if valid_evals:
    y_true = [r['true_label'] for r in valid_evals]
    y_pred = [r['pred_label'] for r in valid_evals]

    accuracy = accuracy_score(y_true, y_pred)

    print(f"--- Final Evaluation: {model} ---")
    print(f"Total Samples: {len(results)}")
    print(f"Valid Conversions: {len(valid_evals)}/{len(results)}")
    print(f"Accuracy: {accuracy:.4f}")
else:
    print(f"Failure: No valid predictions extracted for {model}.")

--- Final Evaluation: mistralai/ministral-14b-2512 ---
Total Samples: 500
Valid Conversions: 194/500
Accuracy: 0.8402


In [33]:
import copy
from sklearn.metrics import accuracy_score

# 1. Define and Run Evaluation
model = 'google/gemma-4-31b-it'
fanout_results[model] = evaluate_model_concurrently(model)

# 2. Scope the extraction to the CURRENT model only
# This prevents the "duplication" bug by ensuring we don't use a stale 'results' variable
raw_data = fanout_results.get(model, [])

if not raw_data:
    print(f"CRITICAL ERROR: No data returned for {model}. Check API connectivity.")
else:
    processed_results = []

    # 3. Process with Forensic Logic (Mapping digits to labels)
    for r in raw_data:
        # Use .copy() to avoid mutating the original data in fanout_results
        entry = r.copy()
        raw_output = str(entry.get('raw_output', '')).strip().lower()

        # Priority 1: Direct Digit Mapping (Matching your Forensic Prompt)
        if '1' in raw_output and '0' not in raw_output:
            entry['pred_label'] = 1
        elif '0' in raw_output and '1' not in raw_output:
            entry['pred_label'] = 0
        # Priority 2: Semantic Fallback
        elif "positive" in raw_output:
            entry['pred_label'] = 1
        elif "negative" in raw_output:
            entry['pred_label'] = 0
        else:
            entry['pred_label'] = -1

        processed_results.append(entry)

    # 4. Filter and Calculate Metrics
    valid_evals = [r for r in processed_results if r['pred_label'] in [0, 1]]

    if valid_evals:
        y_true = [r['true_label'] for r in valid_evals]
        y_pred = [r['pred_label'] for r in valid_evals]

        accuracy = accuracy_score(y_true, y_pred)

        print(f"--- Model: {model} ---")
        print(f"Total Samples: {len(raw_data)}")
        print(f"Valid Binary Predictions: {len(valid_evals)}/{len(raw_data)}")
        print(f"Accuracy: {accuracy:.4f}")

        # Optional: Print a few raw outputs to verify Forensic reasoning
        print("\nSample Forensic Outputs:")
        for sample in processed_results[:3]:
            print(f"Raw: {sample['raw_output']} -> Pred: {sample['pred_label']} (True: {sample['true_label']})")
    else:
        print(f"Extraction Failure: The model {model} did not return '1', '0', or sentiment keywords.")

Starting evaluation for: google/gemma-4-31b-it


google/gemma-4-31b-it:   0%|          | 0/500 [00:00<?, ?it/s]

--- Model: google/gemma-4-31b-it ---
Total Samples: 500
Valid Binary Predictions: 500/500
Accuracy: 0.8300

Sample Forensic Outputs:
Raw: 1 -> Pred: 1 (True: 1)
Raw: 0 -> Pred: 0 (True: 0)
Raw: 1 -> Pred: 1 (True: 0)


In [34]:
import copy
from sklearn.metrics import accuracy_score

# 1. Define and Run Evaluation
model = 'google/gemma-3-4b-it'
fanout_results[model] = evaluate_model_concurrently(model)

# 2. Scope the extraction to the CURRENT model only
# This prevents the "duplication" bug by ensuring we don't use a stale 'results' variable
raw_data = fanout_results.get(model, [])

if not raw_data:
    print(f"CRITICAL ERROR: No data returned for {model}. Check API connectivity.")
else:
    processed_results = []

    # 3. Process with Forensic Logic (Mapping digits to labels)
    for r in raw_data:
        # Use .copy() to avoid mutating the original data in fanout_results
        entry = r.copy()
        raw_output = str(entry.get('raw_output', '')).strip().lower()

        # Priority 1: Direct Digit Mapping (Matching your Forensic Prompt)
        if '1' in raw_output and '0' not in raw_output:
            entry['pred_label'] = 1
        elif '0' in raw_output and '1' not in raw_output:
            entry['pred_label'] = 0
        # Priority 2: Semantic Fallback
        elif "positive" in raw_output:
            entry['pred_label'] = 1
        elif "negative" in raw_output:
            entry['pred_label'] = 0
        else:
            entry['pred_label'] = -1

        processed_results.append(entry)

    # 4. Filter and Calculate Metrics
    valid_evals = [r for r in processed_results if r['pred_label'] in [0, 1]]

    if valid_evals:
        y_true = [r['true_label'] for r in valid_evals]
        y_pred = [r['pred_label'] for r in valid_evals]

        accuracy = accuracy_score(y_true, y_pred)

        print(f"--- Model: {model} ---")
        print(f"Total Samples: {len(raw_data)}")
        print(f"Valid Binary Predictions: {len(valid_evals)}/{len(raw_data)}")
        print(f"Accuracy: {accuracy:.4f}")

        # Optional: Print a few raw outputs to verify Forensic reasoning
        print("\nSample Forensic Outputs:")
        for sample in processed_results[:3]:
            print(f"Raw: {sample['raw_output']} -> Pred: {sample['pred_label']} (True: {sample['true_label']})")
    else:
        print(f"Extraction Failure: The model {model} did not return '1', '0', or sentiment keywords.")

Starting evaluation for: google/gemma-3-4b-it


google/gemma-3-4b-it:   0%|          | 0/500 [00:00<?, ?it/s]

--- Model: google/gemma-3-4b-it ---
Total Samples: 500
Valid Binary Predictions: 500/500
Accuracy: 0.7200

Sample Forensic Outputs:
Raw: 1 -> Pred: 1 (True: 1)
Raw: 1 -> Pred: 1 (True: 0)
Raw: 0 -> Pred: 0 (True: 0)


In [36]:
import copy
from sklearn.metrics import accuracy_score

# 1. Define and Run Evaluation
model = 'mistralai/mistral-7b-instruct-v0.1'
fanout_results[model] = evaluate_model_concurrently(model)

# 2. Scope the extraction to the CURRENT model only
# This prevents the "duplication" bug by ensuring we don't use a stale 'results' variable
raw_data = fanout_results.get(model, [])

if not raw_data:
    print(f"CRITICAL ERROR: No data returned for {model}. Check API connectivity.")
else:
    processed_results = []

    # 3. Process with Forensic Logic (Mapping digits to labels)
    for r in raw_data:
        # Use .copy() to avoid mutating the original data in fanout_results
        entry = r.copy()
        raw_output = str(entry.get('raw_output', '')).strip().lower()

        if '1' in raw_output and '0' not in raw_output:
            entry['pred_label'] = 1
        elif '0' in raw_output and '1' not in raw_output:
            entry['pred_label'] = 0
        else:
            entry['pred_label'] = -1

        processed_results.append(entry)

    # 4. Filter and Calculate Metrics
    valid_evals = [r for r in processed_results if r['pred_label'] in [0, 1]]

    if valid_evals:
        y_true = [r['true_label'] for r in valid_evals]
        y_pred = [r['pred_label'] for r in valid_evals]

        accuracy = accuracy_score(y_true, y_pred)

        print(f"--- Model: {model} ---")
        print(f"Total Samples: {len(raw_data)}")
        print(f"Valid Binary Predictions: {len(valid_evals)}/{len(raw_data)}")
        print(f"Accuracy: {accuracy:.4f}")

        # Optional: Print a few raw outputs to verify Forensic reasoning
        print("\nSample Forensic Outputs:")
        for sample in processed_results[:3]:
            print(f"Raw: {sample['raw_output']} -> Pred: {sample['pred_label']} (True: {sample['true_label']})")
    else:
        print(f"Extraction Failure: The model {model} did not return '1', '0', or sentiment keywords.")

Starting evaluation for: mistralai/mistral-7b-instruct-v0.1


mistralai/mistral-7b-instruct-v0.1:   0%|          | 0/500 [00:00<?, ?it/s]

--- Model: mistralai/mistral-7b-instruct-v0.1 ---
Total Samples: 500
Valid Binary Predictions: 469/500
Accuracy: 0.7676

Sample Forensic Outputs:
Raw: 1 (positive)

explan -> Pred: 1 (True: 0)
Raw: 0. the post expresses a concern for -> Pred: 0 (True: 0)
Raw: 0. the post expresses disappointment or frustration -> Pred: 0 (True: 0)


In [37]:
import copy
from sklearn.metrics import accuracy_score

# 1. Define and Run Evaluation
model = 'mistralai/mistral-nemo'
fanout_results[model] = evaluate_model_concurrently(model)

# 2. Scope the extraction to the CURRENT model only
# This prevents the "duplication" bug by ensuring we don't use a stale 'results' variable
raw_data = fanout_results.get(model, [])

if not raw_data:
    print(f"CRITICAL ERROR: No data returned for {model}. Check API connectivity.")
else:
    processed_results = []

    # 3. Process with Forensic Logic (Mapping digits to labels)
    for r in raw_data:
        # Use .copy() to avoid mutating the original data in fanout_results
        entry = r.copy()
        raw_output = str(entry.get('raw_output', '')).strip().lower()

        if '1' in raw_output and '0' not in raw_output:
            entry['pred_label'] = 1
        elif '0' in raw_output and '1' not in raw_output:
            entry['pred_label'] = 0
        else:
            entry['pred_label'] = -1

        processed_results.append(entry)

    # 4. Filter and Calculate Metrics
    valid_evals = [r for r in processed_results if r['pred_label'] in [0, 1]]

    if valid_evals:
        y_true = [r['true_label'] for r in valid_evals]
        y_pred = [r['pred_label'] for r in valid_evals]

        accuracy = accuracy_score(y_true, y_pred)

        print(f"--- Model: {model} ---")
        print(f"Total Samples: {len(raw_data)}")
        print(f"Valid Binary Predictions: {len(valid_evals)}/{len(raw_data)}")
        print(f"Accuracy: {accuracy:.4f}")

        # Optional: Print a few raw outputs to verify Forensic reasoning
        print("\nSample Forensic Outputs:")
        for sample in processed_results[:3]:
            print(f"Raw: {sample['raw_output']} -> Pred: {sample['pred_label']} (True: {sample['true_label']})")
    else:
        print(f"Extraction Failure: The model {model} did not return '1', '0', or sentiment keywords.")

Starting evaluation for: mistralai/mistral-nemo


mistralai/mistral-nemo:   0%|          | 0/500 [00:00<?, ?it/s]

--- Model: mistralai/mistral-nemo ---
Total Samples: 500
Valid Binary Predictions: 500/500
Accuracy: 0.7820

Sample Forensic Outputs:
Raw: 1 -> Pred: 1 (True: 1)
Raw: 0 -> Pred: 0 (True: 1)
Raw: 0 -> Pred: 0 (True: 0)


In [38]:
import copy
from sklearn.metrics import accuracy_score

# 1. Define and Run Evaluation
model = 'google/gemma-3-12b-it'
fanout_results[model] = evaluate_model_concurrently(model)

# 2. Scope the extraction to the CURRENT model only
# This prevents the "duplication" bug by ensuring we don't use a stale 'results' variable
raw_data = fanout_results.get(model, [])

if not raw_data:
    print(f"CRITICAL ERROR: No data returned for {model}. Check API connectivity.")
else:
    processed_results = []

    # 3. Process with Forensic Logic (Mapping digits to labels)
    for r in raw_data:
        # Use .copy() to avoid mutating the original data in fanout_results
        entry = r.copy()
        raw_output = str(entry.get('raw_output', '')).strip().lower()

        if '1' in raw_output and '0' not in raw_output:
            entry['pred_label'] = 1
        elif '0' in raw_output and '1' not in raw_output:
            entry['pred_label'] = 0
        else:
            entry['pred_label'] = -1

        processed_results.append(entry)

    # 4. Filter and Calculate Metrics
    valid_evals = [r for r in processed_results if r['pred_label'] in [0, 1]]

    if valid_evals:
        y_true = [r['true_label'] for r in valid_evals]
        y_pred = [r['pred_label'] for r in valid_evals]

        accuracy = accuracy_score(y_true, y_pred)

        print(f"--- Model: {model} ---")
        print(f"Total Samples: {len(raw_data)}")
        print(f"Valid Binary Predictions: {len(valid_evals)}/{len(raw_data)}")
        print(f"Accuracy: {accuracy:.4f}")

        # Optional: Print a few raw outputs to verify Forensic reasoning
        print("\nSample Forensic Outputs:")
        for sample in processed_results[:3]:
            print(f"Raw: {sample['raw_output']} -> Pred: {sample['pred_label']} (True: {sample['true_label']})")
    else:
        print(f"Extraction Failure: The model {model} did not return '1', '0', or sentiment keywords.")

Starting evaluation for: google/gemma-3-12b-it


google/gemma-3-12b-it:   0%|          | 0/500 [00:00<?, ?it/s]

--- Model: google/gemma-3-12b-it ---
Total Samples: 500
Valid Binary Predictions: 500/500
Accuracy: 0.7900

Sample Forensic Outputs:
Raw: 0 -> Pred: 0 (True: 1)
Raw: 0 -> Pred: 0 (True: 0)
Raw: 1 -> Pred: 1 (True: 1)


In [40]:
import copy
from sklearn.metrics import accuracy_score

# 1. Define and Run Evaluation
model = 'google/gemma-3-4b-it'
fanout_results[model] = evaluate_model_concurrently(model)

# 2. Scope the extraction to the CURRENT model only
# This prevents the "duplication" bug by ensuring we don't use a stale 'results' variable
raw_data = fanout_results.get(model, [])

if not raw_data:
    print(f"CRITICAL ERROR: No data returned for {model}. Check API connectivity.")
else:
    processed_results = []

    # 3. Process with Forensic Logic (Mapping digits to labels)
    for r in raw_data:
        # Use .copy() to avoid mutating the original data in fanout_results
        entry = r.copy()
        raw_output = str(entry.get('raw_output', '')).strip().lower()

        if '1' in raw_output and '0' not in raw_output:
            entry['pred_label'] = 1
        elif '0' in raw_output and '1' not in raw_output:
            entry['pred_label'] = 0
        else:
            entry['pred_label'] = -1

        processed_results.append(entry)

    # 4. Filter and Calculate Metrics
    valid_evals = [r for r in processed_results if r['pred_label'] in [0, 1]]

    if valid_evals:
        y_true = [r['true_label'] for r in valid_evals]
        y_pred = [r['pred_label'] for r in valid_evals]

        accuracy = accuracy_score(y_true, y_pred)

        print(f"--- Model: {model} ---")
        print(f"Total Samples: {len(raw_data)}")
        print(f"Valid Binary Predictions: {len(valid_evals)}/{len(raw_data)}")
        print(f"Accuracy: {accuracy:.4f}")

        # Optional: Print a few raw outputs to verify Forensic reasoning
        print("\nSample Forensic Outputs:")
        for sample in processed_results[:3]:
            print(f"Raw: {sample['raw_output']} -> Pred: {sample['pred_label']} (True: {sample['true_label']})")
    else:
        print(f"Extraction Failure: The model {model} did not return '1', '0', or sentiment keywords.")

Starting evaluation for: google/gemma-3-4b-it


google/gemma-3-4b-it:   0%|          | 0/500 [00:00<?, ?it/s]

--- Model: google/gemma-3-4b-it ---
Total Samples: 500
Valid Binary Predictions: 500/500
Accuracy: 0.7080

Sample Forensic Outputs:
Raw: error: 429 Client Error: Too Many Requests for url: https://openrouter.ai/api/v1/chat/completions -> Pred: 1 (True: 0)
Raw: error: 429 Client Error: Too Many Requests for url: https://openrouter.ai/api/v1/chat/completions -> Pred: 1 (True: 1)
Raw: error: 429 Client Error: Too Many Requests for url: https://openrouter.ai/api/v1/chat/completions -> Pred: 1 (True: 1)


In [42]:
import copy
from sklearn.metrics import accuracy_score

# 1. Define and Run Evaluation
model = 'mistralai/mistral-7b-instruct-v0.1'
fanout_results[model] = evaluate_model_concurrently(model)

# 2. Scope the extraction to the CURRENT model only
# This prevents the "duplication" bug by ensuring we don't use a stale 'results' variable
raw_data = fanout_results.get(model, [])

if not raw_data:
    print(f"CRITICAL ERROR: No data returned for {model}. Check API connectivity.")
else:
    processed_results = []

    # 3. Process with Forensic Logic (Mapping digits to labels)
    for r in raw_data:
        # Use .copy() to avoid mutating the original data in fanout_results
        entry = r.copy()
        raw_output = str(entry.get('raw_output', '')).strip().lower()

        if '1' in raw_output and '0' not in raw_output:
            entry['pred_label'] = 1
        elif '0' in raw_output and '1' not in raw_output:
            entry['pred_label'] = 0
        else:
            entry['pred_label'] = -1

        processed_results.append(entry)

    # 4. Filter and Calculate Metrics
    valid_evals = [r for r in processed_results if r['pred_label'] in [0, 1]]

    if valid_evals:
        y_true = [r['true_label'] for r in valid_evals]
        y_pred = [r['pred_label'] for r in valid_evals]

        accuracy = accuracy_score(y_true, y_pred)

        print(f"--- Model: {model} ---")
        print(f"Total Samples: {len(raw_data)}")
        print(f"Valid Binary Predictions: {len(valid_evals)}/{len(raw_data)}")
        print(f"Accuracy: {accuracy:.4f}")

        # Optional: Print a few raw outputs to verify Forensic reasoning
        print("\nSample Forensic Outputs:")
        for sample in processed_results[:3]:
            print(f"Raw: {sample['raw_output']} -> Pred: {sample['pred_label']} (True: {sample['true_label']})")
    else:
        print(f"Extraction Failure: The model {model} did not return '1', '0', or sentiment keywords.")

Starting evaluation for: mistralai/mistral-7b-instruct-v0.1


mistralai/mistral-7b-instruct-v0.1:   0%|          | 0/500 [00:00<?, ?it/s]

--- Model: mistralai/mistral-7b-instruct-v0.1 ---
Total Samples: 500
Valid Binary Predictions: 477/500
Accuracy: 0.7673

Sample Forensic Outputs:
Raw: 0. the post expresses disappointment and frustration -> Pred: 0 (True: 0)
Raw: 0. the use of the word "sh -> Pred: 0 (True: 0)
Raw: 0. the post expresses disappointment and sad -> Pred: 0 (True: 0)


Simple prompt

In [5]:
system_instruction = (
"""You are an expert Linguistic Forensic Analyst specializing in early-era social media discourse and sentiment polarity.
Your goal is to decode the emotional intent of high-noise, informal micro-blogging text (Twitter/X).

Analyze the combination of syntax, slang, and punctuation. Output ONLY the integer 1 (Positive) or 0 (Negative).
""")

def evaluate_model_concurrently(model_name):
    print(f"Starting evaluation for: {model_name}")
    results = []

    with ThreadPoolExecutor(max_workers=CONCURRENT_THREADS) as executor:
        future_to_sample = {
            executor.submit(
                call_openrouter,
                model_name,
                f"{system_instruction}\n\n"
                f"""\
Sentiment Polarity Request: Analyze the emotional trajectory of the following micro-blog post.
--------------------------
Post: "{sample['text']}"
--------------------------
Evaluation Guidelines:
- Positive (1): Contentment, humor, excitement, gratitude, or affectionate interaction.
- Negative (0): Boredom, sadness, anger, illness, technical frustration, or loneliness.

Predict the survival of the sentiment as 1 (Positive) or 0 (Negative). Output ONLY the digit.
Prediction:"""
            ): sample for sample in eval_subset
        }

        for future in tqdm(as_completed(future_to_sample), total=NUM_EVAL_SAMPLES, desc=model_name):
            sample = future_to_sample[future]
            prediction = future.result()

            # Map prediction to label (0: Negative, 1: Positive)
            pred_label = 1 if "positive" in prediction else 0 if "negative" in prediction else -1

            results.append({
                "true_label": sample['sentiment'],
                "pred_label": pred_label,
                "raw_output": prediction
            })

    return results

NUM_EVAL_SAMPLES = 500
CONCURRENT_THREADS = 10
SEED=15179996 # reproducible split to get our testing set

eval_subset = dataset["test"].shuffle(seed=SEED).select(range(NUM_EVAL_SAMPLES))


In [8]:
import copy
from sklearn.metrics import accuracy_score

# 1. Define and Run Evaluation
model = 'mistralai/mistral-7b-instruct-v0.1'
fanout_results = {}
fanout_results[model] = evaluate_model_concurrently(model)

# 2. Scope the extraction to the CURRENT model only
# This prevents the "duplication" bug by ensuring we don't use a stale 'results' variable
raw_data = fanout_results.get(model, [])

if not raw_data:
    print(f"CRITICAL ERROR: No data returned for {model}. Check API connectivity.")
else:
    processed_results = []

    # 3. Process with Forensic Logic (Mapping digits to labels)
    for r in raw_data:
        # Use .copy() to avoid mutating the original data in fanout_results
        entry = r.copy()
        raw_output = str(entry.get('raw_output', '')).strip().lower()

        if '1' in raw_output and '0' not in raw_output:
            entry['pred_label'] = 1
        elif '0' in raw_output and '1' not in raw_output:
            entry['pred_label'] = 0
        else:
            entry['pred_label'] = -1

        processed_results.append(entry)

    # 4. Filter and Calculate Metrics
    valid_evals = [r for r in processed_results if r['pred_label'] in [0, 1]]

    if valid_evals:
        y_true = [r['true_label'] for r in valid_evals]
        y_pred = [r['pred_label'] for r in valid_evals]

        accuracy = accuracy_score(y_true, y_pred)

        print(f"--- Model: {model} ---")
        print(f"Total Samples: {len(raw_data)}")
        print(f"Valid Binary Predictions: {len(valid_evals)}/{len(raw_data)}")
        print(f"Accuracy: {accuracy:.4f}")

        # Optional: Print a few raw outputs to verify Forensic reasoning
        print("\nSample Forensic Outputs:")
        for sample in processed_results[:3]:
            print(f"Raw: {sample['raw_output']} -> Pred: {sample['pred_label']} (True: {sample['true_label']})")
    else:
        print(f"Extraction Failure: The model {model} did not return '1', '0', or sentiment keywords.")

Starting evaluation for: mistralai/mistral-7b-instruct-v0.1


mistralai/mistral-7b-instruct-v0.1:   0%|          | 0/500 [00:00<?, ?it/s]

--- Model: mistralai/mistral-7b-instruct-v0.1 ---
Total Samples: 500
Valid Binary Predictions: 499/500
Accuracy: 0.7194

Sample Forensic Outputs:
Raw: 1. -> Pred: 1 (True: 0)
Raw: 1 (positive)

explan -> Pred: 1 (True: 1)
Raw: 1 (positive)

explan -> Pred: 1 (True: 0)
